Import the file

In [ ]:
from google.colab import files

uploaded = files.upload()

uploading the

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import DBSCAN

In [ ]:
df = pd.read_csv("Crime_Data_from_2020_to_Present (1).csv.zip")

print(df.head())

In [ ]:
print(df.columns.tolist())

In [ ]:
print(df.columns.tolist())

In [ ]:
# Paste this and run it
print(df[['LAT', 'LON']].head(10))
print("\nMissing values:")
print(df[['LAT', 'LON']].isnull().sum())

In [ ]:
# Rows where LAT=0 or LON=0 are bad/missing data in this dataset
df_clean = df[(df['LAT'] != 0) & (df['LON'] != 0)].dropna(subset=['LAT', 'LON']).copy()

print(f"Original rows: {len(df)}")
print(f"Clean rows:    {len(df_clean)}")

In [ ]:
from sklearn.preprocessing import StandardScaler

coords = df_clean[['LAT', 'LON']].values

scaler = StandardScaler()
coords_scaled = scaler.fit_transform(coords)

print("Ready! Shape:", coords_scaled.shape)

In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

# Take a sample so it runs fast
sample = coords_scaled[:30000]

nbrs = NearestNeighbors(n_neighbors=5).fit(sample)
distances, _ = nbrs.kneighbors(sample)
distances = np.sort(distances[:, 4])

plt.figure(figsize=(10, 4))
plt.plot(distances)
plt.title("K-Distance Graph — Look for the ELBOW 📍")
plt.xlabel("Points")
plt.ylabel("Distance to 5th neighbour")
plt.grid(True)
plt.show()

In [ ]:
from sklearn.cluster import DBSCAN

sample = coords_scaled[:30000]  # same sample as before

db = DBSCAN(eps=0.2, min_samples=5, n_jobs=-1)
db.fit(sample)
labels = db.labels_

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = list(labels).count(-1)

print(f"✅ Hotspots (clusters) found: {n_clusters}")
print(f"🔇 Noise points (no cluster): {n_noise}")

In [ ]:
# Get real coordinates back
sample_coords = scaler.inverse_transform(sample)

plt.figure(figsize=(12, 8))
plt.scatter(
    sample_coords[:, 1],  # LON
    sample_coords[:, 0],  # LAT
    c=labels,
    cmap='tab20',
    s=1,
    alpha=0.5
)
plt.title("🗺️ Crime Hotspots in LA (DBSCAN)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.colorbar(label="Cluster / Hotspot")
plt.show()

In [ ]:
from sklearn.cluster import DBSCAN

sample = coords_scaled[:30000]

db = DBSCAN(eps=0.05, min_samples=5, n_jobs=-1)  # changed 0.2 → 0.05
db.fit(sample)
labels = db.labels_

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = list(labels).count(-1)

print(f"✅ Hotspots found: {n_clusters}")
print(f"🔇 Noise points:  {n_noise}")

In [ ]:
import pandas as pd

results = pd.DataFrame({'Cluster': labels})
top_hotspots = results[results['Cluster'] != -1]['Cluster'].value_counts().head(10)

print("🔴 TOP 10 Crime Hotspots")
print("="*30)
for rank, (hotspot, count) in enumerate(top_hotspots.items(), 1):
    print(f"#{rank:>2}  Hotspot {hotspot:>3}  →  {count} crimes")

In [ ]:
import folium

# Get real coordinates back
sample_coords = scaler.inverse_transform(sample)

# Centre map on LA
m = folium.Map(location=[34.05, -118.25], zoom_start=11)

# Plot each crime point (skip noise = -1)
colors = ['red','blue','green','purple','orange','darkred',
          'lightred','beige','darkblue','darkgreen']

for i, (lat, lon) in enumerate(sample_coords):
    cluster = labels[i]
    if cluster == -1:
        continue  # skip noise
    folium.CircleMarker(
        location=[lat, lon],
        radius=3,
        color=colors[cluster % len(colors)],
        fill=True,
        fill_opacity=0.6,
        popup=f"Hotspot {cluster}"
    ).add_to(m)

# Save the map
m.save("crime_hotspots_map.html")
print("✅ Map saved! Download crime_hotspots_map.html to view it")
m  # This shows the map inline in Colab!

In [ ]:
from google.colab import files
files.download("crime_hotspots_map.html")

In [ ]:
total_points = len(labels)
total_clustered = total_points - n_noise
noise_pct = (n_noise / total_points) * 100
top1_pct = (20303 / total_clustered) * 100

print("=" * 40)
print("      DBSCAN CRIME ANALYSIS SUMMARY")
print("=" * 40)
print(f"  Total crime incidents analysed : {total_points:,}")
print(f"  Crime hotspots identified      : {n_clusters}")
print(f"  Crimes in hotspots             : {total_clustered:,}")
print(f"  Isolated incidents (noise)     : {n_noise} ({noise_pct:.1f}%)")
print(f"  Largest hotspot crimes         : 20,303")
print(f"  Top 2 hotspots cover           : {((20303+5367)/total_clustered*100):.1f}% of all crimes")
print("=" * 40)

In [ ]:
from folium.plugins import HeatMap
import folium

sample_coords = scaler.inverse_transform(sample)

# Only use 3000 points max
heat_data = [
    [lat, lon]
    for (lat, lon), label in zip(sample_coords[:3000], labels[:3000])
    if label != -1
]

m3 = folium.Map(location=[34.05, -118.25], zoom_start=11)
HeatMap(heat_data, radius=10).add_to(m3)
m3.save("heatmap.html")

from google.colab import files
files.download("heatmap.html")
print("Done!")
